In [1]:
""" ripped from: https://github.com/stegmaja/black-hole-spin-orbit-tilts/blob/main/main.ipynb """
import os
import pickle
from argparse import ArgumentParser

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import bilby as bb
from bilby.hyper.model import Model
from bilby.core.prior import PriorDict
from bilby.core.prior import ConditionalPriorDict
from bilby.core.prior import Uniform
from bilby.core.prior import TruncatedNormal
from bilby.core.prior import DirichletElement

import gwpopulation as gwpop
from gwpopulation.experimental.jax import JittedLikelihood
gwpop.set_backend("jax")

# TODO: cleanup
xp = gwpop.utils.xp
import jax
import jax.numpy as jnp

import h5ify
from util import scan
from util import plot_corner
from util import write_config

from pixelpop.models.gwpop_models import PowerlawPlusPeak_MassRatio
from models import BrokenPowerlawPlusTwoPeaks_PrimaryMass_FullSmooth
from models import bpl2p_m1q

/work/submit/newolfe/miniforge3/envs/just-for-kicks-260228/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from data import get_data
_, posteriors, injections, ln_evidences = get_data(
    snr_thresh=10,
    far_thresh=1,
    prefer_xphm=False,
    prefer_xphm_gwtc3=True,
    return_ln_evidence=True
)

if 'log_prior' in posteriors:
    posteriors['prior'] = xp.exp(posteriors['log_prior'])
if 'log_prior' in injections:
    injections['prior'] = xp.exp(injections['log_prior'])

# format expected by gwpop
posteriors = [
    pd.DataFrame.from_dict(
        {k : v[i] for k, v in posteriors.items()},
        orient='columns'
    )
    for i in range(posteriors['prior'].shape[0])
]

data will be saved to/loaded from ../../data/lvk/GWTC-4-mass_1_source-mass_ratio-redshift-a_1-a_2-cos_tilt_1-cos_tilt_2/posteriors-xphm-gwtc3.h5
loading posteriors from ../../data/lvk/GWTC-4-mass_1_source-mass_ratio-redshift-a_1-a_2-cos_tilt_1-cos_tilt_2/posteriors-xphm-gwtc3.h5


In [3]:
def get_model(model):
    from models import default_stegmann_spin_model

    def spin_model(
        dataset,
        mu_1,
        sigma_1,
        mu_tilt_1,
        sigma_tilt_1,
        mu_2,
        sigma_2,
        mu_3,
        sigma_3,
        weight_a,
        m_cut,
    ):
        return default_stegmann_spin_model(
            dataset,
            mu_1,
            sigma_1,
            mu_tilt_1,
            sigma_tilt_1,
            mu_2,
            sigma_2,
            mu_3,
            sigma_3,
            weight_a,
            m_cut,
            stable_expit=stable_expit
        )

    if model == 'default-spin-simple-power-law-mass':
        model_functions = [
            gwpop.models.mass.two_component_primary_mass_ratio,
            spin_model,
        ]
    elif model == 'default-spin-bpl2p-mass':
        model_functions = [
            bpl2p_m1q,
            spin_model,
        ]
    elif model == 'twomass':
        from models import twomass_and_spin_model
        model_functions = [twomass_and_spin_model]
    elif model == 'threemass':
        from models import threemass_and_spin_model
        model_functions = [threemass_and_spin_model]
    else:
        raise ValueError(f'bad model {model}')

    model_functions += [
        gwpop.models.redshift.PowerLawRedshift(cosmo_model="Planck15")
    ]

    return Model(
        model_functions=model_functions,
        cache=False,
    )

In [4]:
def make_mu_conversion(order):
    """Return conversion_function for HyperparameterLikelihood."""
    if order == 'none':
        return lambda params: (params, [])
    def convert(parameters):
        g0, g1 = parameters['g_0'], parameters['g_1']
        if order == 'ascending':
            parameters['mu_1'], parameters['mu_2'] = g0, g0 + g1
        else:  # descending
            parameters['mu_1'], parameters['mu_2'] = g0 + g1, g0
        return parameters, ['mu_1', 'mu_2']
    return convert

In [8]:
model = 'twomass'
maximum_uncertainty = 1
constrain_mu_order = 'none'

In [9]:
vt = gwpop.vt.ResamplingVT(
    model=get_model(model),
    data=injections,
    n_events=len(posteriors)
)

# set random state for re-sampling the single-event PE
np.random.seed(42)

likelihood = gwpop.hyperpe.HyperparameterLikelihood(
    posteriors=posteriors,
    hyper_prior=get_model(model),
    selection_function=vt,
    maximum_uncertainty=maximum_uncertainty,
    ln_evidences=ln_evidences,
    conversion_function=make_mu_conversion(constrain_mu_order),
)

In [11]:
priors = './priors/twomass-only-mass.prior'

if model == 'default-spin-simple-power-law-mass':
    priors = PriorDict()
    priors["alpha"] = Uniform(minimum=-2, maximum=4, latex_label="$\\alpha$")
    priors["beta"] = Uniform(minimum=-4, maximum=12, latex_label="$\\beta$")
    priors["mmin"] = Uniform(minimum=2, maximum=2.5, latex_label="$m_{\\min}$")
    priors["mmax"] = Uniform(minimum=80, maximum=100, latex_label="$m_{\\max}$")
    priors["lam"] = Uniform(minimum=0, maximum=1, latex_label="$\\lambda_{m}$")
    priors["mpp"] = Uniform(minimum=10, maximum=50, latex_label="$\\mu_{m}$")
    priors["sigpp"] = Uniform(minimum=1, maximum=10, latex_label="$\\sigma_{m}$")
    priors["gaussian_mass_maximum"] = 100
else:
    priors = ConditionalPriorDict(priors)

# spin
if constrain_mu_order == 'none':
    priors["mu_1"] = Uniform(minimum=0, maximum=1, latex_label="$\\mu_1$")
    priors["mu_2"] = Uniform(minimum=0, maximum=1, latex_label="$\\mu_2$")
else:
    priors["g_0"] = DirichletElement(order=0, n_dimensions=3, label='g_')
    priors["g_1"] = DirichletElement(order=1, n_dimensions=3, label='g_')

priors["sigma_1"] = Uniform(minimum=0.1, maximum=1, latex_label="$\\sigma_1$")
priors["mu_tilt_1"] = Uniform(minimum=-1, maximum=1, latex_label="$\\mu_{t,1}$")
priors["sigma_tilt_1"] = TruncatedNormal(minimum=0.1, maximum=4, sigma=1/2, mu=0, latex_label="$\\sigma_{t,1}$")
priors["sigma_2"] = Uniform(minimum=0.1, maximum=1, latex_label="$\\sigma_2$")
priors["weight_a"] = Uniform(minimum=0, maximum=1, latex_label="$w_a$")
priors["mu_3"] = Uniform(minimum=0, maximum=1, latex_label="$\\mu_3$")
priors["sigma_3"] = Uniform(minimum=0.1, maximum=1, latex_label="$\\sigma_3$")
priors["m_cut"] = Uniform(minimum=10, maximum=100, latex_label="$m_{\\rm cut}$")

# redshift
priors["lamb"] = Uniform(minimum=-1, maximum=10, latex_label="$\\lambda_{z}$")

In [ ]:
jit_likelihood = JittedLikelihood(likelihood)
ll = jit_likelihood.log_likelihood_ratio(priors.sample())